# Feature Engineering — Churn Prediction (Telco Customer Churn)

Parte del EDA (`01_eda.ipynb`): limpieza, encoding y escalado.
Objetivo: dejar un dataset limpio (`processed/telco_churn_clean.csv`) y un `ColumnTransformer` reutilizable, consistente con `src/train.py`, listo para `03_model_training.ipynb`..

## 1. Carga de datos crudos

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", None)


In [2]:
# Opción A: descargar desde Kaggle con kagglehub
import kagglehub
import glob, os

path = kagglehub.dataset_download("blastchar/telco-customer-churn")
csv_path = glob.glob(os.path.join(path, "*.csv"))[0]
df = pd.read_csv(csv_path)
df.shape


Using Colab cache for faster access to the 'telco-customer-churn' dataset.


(7043, 21)

Alternativa: leer el crudo ya subido a S3:
```python
import sys
sys.path.append("../src")
from s3_utils import read_csv_from_s3

df = read_csv_from_s3("raw/telco_churn.csv")
```

## 2. Limpieza básica

Pasos identificados en el EDA:
- `TotalCharges` a numérico (los strings vacíos se convierten en NaN)
- Filas nulas resultantes coinciden con `tenure == 0` → se imputan con 0, ya que no hubo facturación
- `customerID` se descarta: es un identificador, no aporta señal predictiva

In [3]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Confirmar que los nulos coinciden con tenure == 0
assert (df.loc[df["TotalCharges"].isnull(), "tenure"] == 0).all()

df["TotalCharges"] = df["TotalCharges"].fillna(0)
df = df.drop(columns=["customerID"])
df.shape


(7043, 20)

## 3. Variable target

Se convierte `Churn` de `Yes`/`No` a `1`/`0` para que sea compatible con las métricas de sklearn (`roc_auc_score`, `f1_score`, etc.), igual que en `src/train.py`.

In [4]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
df["Churn"].value_counts(normalize=True)


,proportion
Churn,
0,0.73463
1,0.26537


## 4. `SeniorCitizen` como categórica

Viene codificada como `0`/`1` (tipo numérico) pero es conceptualmente binaria/categórica. Se convierte a `Yes`/`No` para que el `OneHotEncoder` la trate igual que el resto de las binarias (`Partner`, `Dependents`, etc.) y no la escale como si fuera una magnitud continua.

In [5]:
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
df["SeniorCitizen"].value_counts()


,count
SeniorCitizen,
No,5901
Yes,1142


## 5. Separar features numéricas y categóricas

Mismo criterio que `build_preprocessor()` en `src/train.py`: se detectan automáticamente por tipo de dato, para que el notebook y el script de entrenamiento queden alineados.

In [6]:
TARGET_COL = "Churn"

features = df.drop(columns=[TARGET_COL])
numeric_cols = features.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = features.select_dtypes(include=["object"]).columns.tolist()

print("Numéricas:", numeric_cols)
print("Categóricas:", categorical_cols)


Numéricas: ['tenure', 'MonthlyCharges', 'TotalCharges']
Categóricas: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## 6. Train/test split

Split estratificado por `Churn` para mantener la proporción de clases (dataset desbalanceado) tanto en train como en test.

In [7]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape


((5634, 19), (1409, 19))

## 7. Preprocesador: escalado + one-hot encoding

`StandardScaler` para numéricas, `OneHotEncoder` para categóricas. Se ajusta (`fit`) solo sobre train para evitar fuga de información (data leakage) del test set.

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

X_train_transformed.shape, X_test_transformed.shape


((5634, 46), (1409, 46))

## 8. Nombres de columnas post-encoding (referencia)

Útil más adelante para interpretar coeficientes o importancias de features.

In [14]:
feature_names = preprocessor.get_feature_names_out()
len(feature_names), feature_names[:15]


(46,
 array(['num__tenure', 'num__MonthlyCharges', 'num__TotalCharges',
        'cat__gender_Female', 'cat__gender_Male', 'cat__SeniorCitizen_No',
        'cat__SeniorCitizen_Yes', 'cat__Partner_No', 'cat__Partner_Yes',
        'cat__Dependents_No', 'cat__Dependents_Yes',
        'cat__PhoneService_No', 'cat__PhoneService_Yes',
        'cat__MultipleLines_No', 'cat__MultipleLines_No phone service'],
       dtype=object))

## 9. Guardar dataset limpio (pre-encoding) en S3

Se sube la versión limpia pero **sin encodear** — el encoding/escalado se re-ajusta dentro del `Pipeline` de `src/train.py` para que quede autocontenido y no dependa de artefactos externos.

In [18]:
!pip install boto3

import os
import io
import boto3
import pandas as pd

AWS_ACCESS_KEY_ID = "YOUR_AWS_ACCESS_KEY"
AWS_SECRET_ACCESS_KEY = "YOUR_AWS_SECRET_KEY"
AWS_REGION = "us-east-2"
BUCKET_NAME = "mi-churn-project-2026"

def upload_dataframe_as_csv(df: pd.DataFrame, s3_key: str, bucket: str = BUCKET_NAME):
    s3 = boto3.client(
        "s3",
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
        region_name=AWS_REGION,
    )
    buffer = io.StringIO()
    df.to_csv(buffer, index=False)
    s3.put_object(Bucket=bucket, Key=s3_key, Body=buffer.getvalue())
    print(f"DataFrame subido con éxito a s3://{bucket}/{s3_key}")

# Guardar dataset limpio en S3
df_clean = pd.concat([X, y], axis=1)
upload_dataframe_as_csv(df_clean, "processed/telco_churn_clean.csv")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 1.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 50.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.7 MB/s eta 0:00:00
DataFrame subido con éxito a s3://mi-churn-project-2026/processed/telco_churn_clean.csv


## 10. Notas y próximos pasos

- El dataset limpio (`processed/telco_churn_clean.csv`) queda en S3, listo para ser leído directamente por `03_model_training.ipynb` o por `src/train.py`.
- El `ColumnTransformer` **no se guarda por separado**: `src/train.py` reconstruye el mismo preprocesador dentro de un `Pipeline` junto con el clasificador, para que el modelo final sea un único artefacto autocontenido (`models/churn_best_model.joblib`) y no haya que sincronizar dos archivos distintos al desplegar.
- Siguiente paso: `03_model_training.ipynb` — entrenar y comparar Logistic Regression, Random Forest y XGBoost sobre `df_clean`, replicando la lógica de `train_and_compare()` en `src/train.py`.